# Experimento: VCP con Trend Template + Drawdown

Extiende el experimento original (`run_vcp_experiment.ipynb`) con dos mejoras:

1. **Trend Template como filtro previo**: una señal VCP solo se confirma si el activo
   cumple las 7 condiciones de la Plantilla de Tendencia de Minervini (Etapa 2) en la
   fecha de la señal. Señales que no cumplen se descartan.

2. **Métricas de drawdown**:
   - **Max drawdown por trade**: caída máxima desde el pico de equity durante cada operación.
   - **Max drawdown del activo**: caída máxima del precio del activo en todo el periodo disponible.

Se mantiene la grilla de `volume_ratio_threshold = [1.0, 1.5, 2.0]` del experimento original.

In [ ]:
import sys
import tempfile
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import mlflow

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from models.configs import ATRZigZagConfig
from vcp_detection.heuristic import ATRZigZagDetector, run_full_vcp_pipeline
from vcp_detection.analysis import (
    group_signals_into_patterns,
    simulate_trade,
    plot_vcp_pattern,
    plot_trade_simulation,
)
from stages.trend_template import evaluate_trend_template

pd.set_option("display.float_format", "{:.4f}".format)
print("Imports OK")

## 1. Configuración

In [ ]:
# --- Grilla de experimento ---
VOLUME_RATIO_THRESHOLDS = [1.0, 1.5, 2.0]

# --- Parámetros fijos ---
SWING_CONFIG = ATRZigZagConfig(atr_length=14, atr_mult=2.0, use_close_only=False)

SEQUENCE_PARAMS = {
    "method": "tolerance",
    "min_contractions": 2,
    "max_contractions": 6,
    "lookback_bars": 126,
    "tolerance": 0.10,
    "max_depth_pct": 0.35,
    "min_total_reduction": 0.80,
    "max_gap_between_contractions_days": None,
}

COMPRESSION_PARAMS = {
    "method": "ratio",
    "atr_period": 14,
    "ratio_threshold": 0.85,
}

VOLUME_CONTRACTION_PARAMS = {
    "method": "ratio",
    "volume_column": "volume",
    "ratio_threshold": 0.85,
}

RISK_PARAMS = {
    "max_stop_loss_pct": 0.07,
    "breakeven_r_multiple": 2.0,
    "trailing_sma_period": 20,
    "trailing_volume_factor": 1.5,
}

# --- Tickers ---
DATA_DIR = project_root / "data" / "csv"
TICKERS = sorted([p.stem for p in DATA_DIR.glob("*.csv")])
print(f"Tickers ({len(TICKERS)}): {TICKERS}")
print(f"Grilla: volume_ratio_threshold = {VOLUME_RATIO_THRESHOLDS}")
print(f"Total corridas: {len(VOLUME_RATIO_THRESHOLDS)} configs x {len(TICKERS)} tickers = {len(VOLUME_RATIO_THRESHOLDS) * len(TICKERS)}")

## 2. Funciones auxiliares

In [ ]:
def load_ohlc(ticker: str) -> pd.DataFrame:
    path = DATA_DIR / f"{ticker}.csv"
    return pd.read_csv(path, parse_dates=["date"], index_col="date")


def compute_trade_max_drawdown(ohlc: pd.DataFrame, entry_date, exit_date) -> float:
    """Max drawdown durante un trade (desde el pico running, empezando en entry)."""
    trade_closes = ohlc.loc[entry_date:exit_date, "close"]
    if len(trade_closes) < 2:
        return 0.0
    running_peak = trade_closes.cummax()
    drawdown_series = (trade_closes - running_peak) / running_peak
    return float(drawdown_series.min())


def compute_asset_max_drawdown(ohlc: pd.DataFrame) -> float:
    """Max drawdown del activo en todo el periodo disponible."""
    close = ohlc["close"]
    running_peak = close.cummax()
    drawdown_series = (close - running_peak) / running_peak
    return float(drawdown_series.min())


def run_ticker_analysis(
    ticker: str,
    ohlc: pd.DataFrame,
    template_df: pd.DataFrame,
    breakout_params: dict,
    risk_params: dict,
) -> dict:
    """Pipeline VCP completo con filtro de trend template + métricas de drawdown."""
    swing_detector = ATRZigZagDetector(SWING_CONFIG)

    results = run_full_vcp_pipeline(
        ohlc=ohlc,
        swing_detector=swing_detector,
        sequence_params=SEQUENCE_PARAMS,
        compression_params=COMPRESSION_PARAMS,
        breakout_params=breakout_params,
        volume_contraction_params=VOLUME_CONTRACTION_PARAMS,
    )

    # Señales VCP sin filtro
    signals_raw = {dt: sig for dt, sig in results.items() if sig is not None}

    # Filtrar por trend template: solo confirmar si el activo está en Etapa 2
    all_signals = {}
    n_filtered = 0
    for dt, sig in signals_raw.items():
        if dt in template_df.index and bool(template_df.loc[dt, "trend_template"]):
            all_signals[dt] = sig
        else:
            n_filtered += 1

    patterns = group_signals_into_patterns(all_signals, risk_params=risk_params)

    trades = []
    for p in patterns:
        trade = simulate_trade(ohlc, p, risk_params)
        trade["pattern"] = p
        trade["max_drawdown"] = compute_trade_max_drawdown(
            ohlc, p["first_signal_date"], trade["exit_date"],
        )
        trades.append(trade)

    asset_max_dd = compute_asset_max_drawdown(ohlc)

    n_trades = len(trades)
    if n_trades > 0:
        wins = sum(1 for t in trades if t["pnl_pct"] > 0)
        winrate = wins / n_trades
        cumulative_return = float(np.prod([1 + t["pnl_pct"] for t in trades]) - 1)
        avg_r_multiple = float(np.mean([t["r_multiple"] for t in trades]))
        worst_trade_dd = float(min(t["max_drawdown"] for t in trades))
        avg_trade_dd = float(np.mean([t["max_drawdown"] for t in trades]))
    else:
        winrate = 0.0
        cumulative_return = 0.0
        avg_r_multiple = 0.0
        worst_trade_dd = 0.0
        avg_trade_dd = 0.0

    return {
        "signals": all_signals,
        "patterns": patterns,
        "trades": trades,
        "metrics": {
            "n_signals_raw": len(signals_raw),
            "n_signals": len(all_signals),
            "n_filtered_by_template": n_filtered,
            "n_patterns": len(patterns),
            "n_trades": n_trades,
            "winrate": winrate,
            "cumulative_return": cumulative_return,
            "avg_r_multiple": avg_r_multiple,
            "worst_trade_drawdown": worst_trade_dd,
            "avg_trade_drawdown": avg_trade_dd,
            "asset_max_drawdown": asset_max_dd,
        },
    }


def build_trade_table(trades: list[dict], ticker: str) -> pd.DataFrame:
    rows = []
    for i, t in enumerate(trades, 1):
        rows.append({
            "ticker": ticker,
            "trade_num": i,
            "entry_date": t["pattern"]["first_signal_date"].strftime("%Y-%m-%d"),
            "exit_date": t["exit_date"].strftime("%Y-%m-%d"),
            "exit_reason": t["exit_reason"],
            "duration_days": t["duration_days"],
            "entry_price": t["pattern"]["entry_price"],
            "exit_price": t["exit_price"],
            "pnl_pct": t["pnl_pct"],
            "r_multiple": t["r_multiple"],
            "max_r": t["max_r"],
            "max_drawdown": t["max_drawdown"],
            "stop_method": t["pattern"]["stop_method"],
            "n_contractions": t["pattern"]["n_contractions"],
            "atr_ratio": t["pattern"]["atr_ratio"],
        })
    return pd.DataFrame(rows)


print("Funciones auxiliares definidas.")

## 3. Setup MLflow

In [ ]:
EXPERIMENT_NAME = "VCP_TrendTemplate_Drawdown"
mlflow.set_tracking_uri(str(project_root / "mlruns"))
mlflow.set_experiment(EXPERIMENT_NAME)
print(f"MLflow experiment: {EXPERIMENT_NAME}")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")

## 4. Pre-cómputo: Trend Template por ticker

Evalúa la Plantilla de Tendencia una sola vez por ticker (no depende del threshold).
Se reutiliza en todas las configuraciones.

In [ ]:
template_cache = {}
template_summary = []

for ticker in TICKERS:
    ohlc = load_ohlc(ticker)
    tpl = evaluate_trend_template(ohlc)
    template_cache[ticker] = tpl

    n_days = len(tpl)
    n_stage2 = int(tpl["trend_template"].sum())
    pct = n_stage2 / n_days if n_days > 0 else 0.0

    template_summary.append({
        "ticker": ticker,
        "total_days": n_days,
        "days_in_stage2": n_stage2,
        "pct_in_stage2": pct,
    })

template_summary_df = pd.DataFrame(template_summary).set_index("ticker")
print("Trend Template evaluado para todos los tickers:\n")
display(template_summary_df.style.format({
    "pct_in_stage2": "{:.1%}",
}).background_gradient(subset=["pct_in_stage2"], cmap="RdYlGn", vmin=0, vmax=1))

## 5. Ejecución del experimento

Loop principal: por cada `volume_ratio_threshold` se crea un **parent run** en MLflow.
Dentro, por cada ticker un **child run** con métricas, plots, tablas y drawdown.

In [ ]:
all_summaries = []

ALL_PARAMS = {
    "atr_length": SWING_CONFIG.atr_length,
    "atr_mult": SWING_CONFIG.atr_mult,
    "use_close_only": SWING_CONFIG.use_close_only,
    "seq_method": SEQUENCE_PARAMS["method"],
    "min_contractions": SEQUENCE_PARAMS["min_contractions"],
    "max_contractions": SEQUENCE_PARAMS["max_contractions"],
    "lookback_bars": SEQUENCE_PARAMS["lookback_bars"],
    "tolerance": SEQUENCE_PARAMS["tolerance"],
    "max_depth_pct": SEQUENCE_PARAMS["max_depth_pct"],
    "min_total_reduction": SEQUENCE_PARAMS["min_total_reduction"],
    "compression_method": COMPRESSION_PARAMS["method"],
    "compression_threshold": COMPRESSION_PARAMS["ratio_threshold"],
    "vol_contraction_method": VOLUME_CONTRACTION_PARAMS["method"],
    "vol_contraction_threshold": VOLUME_CONTRACTION_PARAMS["ratio_threshold"],
    "max_stop_loss_pct": RISK_PARAMS["max_stop_loss_pct"],
    "breakeven_r_multiple": RISK_PARAMS["breakeven_r_multiple"],
    "trailing_sma_period": RISK_PARAMS["trailing_sma_period"],
    "trailing_volume_factor": RISK_PARAMS["trailing_volume_factor"],
    "trend_template_filter": True,
}

for threshold in VOLUME_RATIO_THRESHOLDS:
    breakout_params = {
        "volume_method": "ratio",
        "volume_ratio_threshold": threshold,
        "volume_lookback_days": 50,
        "require_volume_confirmation": True,
    }

    config_name = f"threshold_{threshold}"
    print(f"\n{'='*70}")
    print(f"CONFIG: {config_name} (volume_ratio_threshold={threshold})")
    print(f"{'='*70}")

    with mlflow.start_run(run_name=config_name) as parent_run:
        mlflow.log_params({
            **ALL_PARAMS,
            "volume_ratio_threshold": threshold,
            "volume_lookback_days": breakout_params["volume_lookback_days"],
        })

        ticker_summaries = []
        all_trades_for_config = []

        for ticker in TICKERS:
            print(f"  {ticker}...", end=" ")
            ohlc = load_ohlc(ticker)
            template_df = template_cache[ticker]

            with mlflow.start_run(run_name=ticker, nested=True) as child_run:
                mlflow.log_params({
                    **ALL_PARAMS,
                    "ticker": ticker,
                    "volume_ratio_threshold": threshold,
                    "volume_lookback_days": breakout_params["volume_lookback_days"],
                    "n_bars": len(ohlc),
                })

                analysis = run_ticker_analysis(
                    ticker, ohlc, template_df, breakout_params, RISK_PARAMS,
                )
                metrics = analysis["metrics"]

                mlflow.log_metrics({
                    "n_signals_raw": metrics["n_signals_raw"],
                    "n_signals": metrics["n_signals"],
                    "n_filtered_by_template": metrics["n_filtered_by_template"],
                    "n_patterns": metrics["n_patterns"],
                    "n_trades": metrics["n_trades"],
                    "winrate": metrics["winrate"],
                    "cumulative_return": metrics["cumulative_return"],
                    "avg_r_multiple": metrics["avg_r_multiple"],
                    "worst_trade_drawdown": metrics["worst_trade_drawdown"],
                    "avg_trade_drawdown": metrics["avg_trade_drawdown"],
                    "asset_max_drawdown": metrics["asset_max_drawdown"],
                })

                with tempfile.TemporaryDirectory() as tmpdir:
                    for j, pattern in enumerate(analysis["patterns"], 1):
                        plot_path = Path(tmpdir) / f"pattern_{j}.png"
                        plot_vcp_pattern(
                            ohlc, pattern, pattern_number=j,
                            ticker=ticker, save_path=str(plot_path),
                        )
                        mlflow.log_artifact(str(plot_path), "pattern_plots")

                    for j, (pat, trade) in enumerate(
                        zip(analysis["patterns"], analysis["trades"]), 1
                    ):
                        trade_plot_path = Path(tmpdir) / f"trade_{j}.png"
                        plot_trade_simulation(
                            ohlc, pat, trade, pattern_number=j,
                            risk_params=RISK_PARAMS, ticker=ticker,
                            save_path=str(trade_plot_path),
                        )
                        mlflow.log_artifact(str(trade_plot_path), "trade_plots")

                    if analysis["trades"]:
                        trade_df = build_trade_table(analysis["trades"], ticker)
                        trade_csv_path = Path(tmpdir) / "trades.csv"
                        trade_df.to_csv(trade_csv_path, index=False)
                        mlflow.log_artifact(str(trade_csv_path), "tables")
                        all_trades_for_config.append(trade_df)

                ticker_summaries.append({
                    "ticker": ticker,
                    "n_signals_raw": metrics["n_signals_raw"],
                    "n_signals": metrics["n_signals"],
                    "n_filtered": metrics["n_filtered_by_template"],
                    "n_patterns": metrics["n_patterns"],
                    "n_trades": metrics["n_trades"],
                    "winrate": metrics["winrate"],
                    "cumulative_return": metrics["cumulative_return"],
                    "avg_r_multiple": metrics["avg_r_multiple"],
                    "worst_trade_drawdown": metrics["worst_trade_drawdown"],
                    "avg_trade_drawdown": metrics["avg_trade_drawdown"],
                    "asset_max_drawdown": metrics["asset_max_drawdown"],
                })

                print(
                    f"{metrics['n_signals_raw']} raw -> {metrics['n_signals']} filtered, "
                    f"{metrics['n_trades']} trades, "
                    f"WR={metrics['winrate']:.0%}, "
                    f"CR={metrics['cumulative_return']:+.1%}, "
                    f"DD={metrics['worst_trade_drawdown']:+.1%}"
                )

        summary_df = pd.DataFrame(ticker_summaries)
        summary_df["volume_ratio_threshold"] = threshold
        all_summaries.append(summary_df)

        tickers_with_trades = summary_df[summary_df["n_trades"] > 0]

        agg_metrics = {
            "total_signals_raw": int(summary_df["n_signals_raw"].sum()),
            "total_signals_filtered": int(summary_df["n_signals"].sum()),
            "total_filtered_by_template": int(summary_df["n_filtered"].sum()),
            "total_patterns": int(summary_df["n_patterns"].sum()),
            "total_trades": int(summary_df["n_trades"].sum()),
            "tickers_with_patterns": int((summary_df["n_patterns"] > 0).sum()),
            "tickers_with_trades": int((summary_df["n_trades"] > 0).sum()),
            "avg_winrate": float(tickers_with_trades["winrate"].mean())
            if len(tickers_with_trades) > 0 else 0.0,
            "median_winrate": float(tickers_with_trades["winrate"].median())
            if len(tickers_with_trades) > 0 else 0.0,
            "avg_cumulative_return": float(tickers_with_trades["cumulative_return"].mean())
            if len(tickers_with_trades) > 0 else 0.0,
            "avg_r_multiple": float(tickers_with_trades["avg_r_multiple"].mean())
            if len(tickers_with_trades) > 0 else 0.0,
            "worst_trade_drawdown": float(tickers_with_trades["worst_trade_drawdown"].min())
            if len(tickers_with_trades) > 0 else 0.0,
            "avg_trade_drawdown": float(tickers_with_trades["avg_trade_drawdown"].mean())
            if len(tickers_with_trades) > 0 else 0.0,
        }
        mlflow.log_metrics(agg_metrics)

        with tempfile.TemporaryDirectory() as tmpdir:
            summary_path = Path(tmpdir) / f"summary_{config_name}.csv"
            summary_df.to_csv(summary_path, index=False)
            mlflow.log_artifact(str(summary_path), "summaries")

            if all_trades_for_config:
                all_trades_df = pd.concat(all_trades_for_config, ignore_index=True)
                all_trades_path = Path(tmpdir) / f"all_trades_{config_name}.csv"
                all_trades_df.to_csv(all_trades_path, index=False)
                mlflow.log_artifact(str(all_trades_path), "summaries")

        print(
            f"\n  AGREGADO: {agg_metrics['total_signals_raw']} señales raw, "
            f"{agg_metrics['total_filtered_by_template']} filtradas por template, "
            f"{agg_metrics['total_trades']} trades, "
            f"avg WR={agg_metrics['avg_winrate']:.0%}, "
            f"avg CR={agg_metrics['avg_cumulative_return']:+.1%}, "
            f"worst DD={agg_metrics['worst_trade_drawdown']:+.1%}"
        )

print("\nExperimento completo!")

## 6. Comparación entre configuraciones

Tablas pivoteadas: winrate, retorno acumulado, y drawdown por ticker para cada threshold.
Se muestran también las señales filtradas por el Trend Template.

In [ ]:
comparison_df = pd.concat(all_summaries, ignore_index=True)

# --- Señales filtradas por Trend Template ---
pivot_raw = comparison_df.pivot(index="ticker", columns="volume_ratio_threshold", values="n_signals_raw")
pivot_filtered = comparison_df.pivot(index="ticker", columns="volume_ratio_threshold", values="n_signals")

print("=== Señales VCP: Raw vs Filtradas por Trend Template (threshold=1.5) ===")
filter_comparison = pd.DataFrame({
    "raw": pivot_raw[1.5] if 1.5 in pivot_raw.columns else pivot_raw.iloc[:, 0],
    "con_template": pivot_filtered[1.5] if 1.5 in pivot_filtered.columns else pivot_filtered.iloc[:, 0],
})
filter_comparison["descartadas"] = filter_comparison["raw"] - filter_comparison["con_template"]
filter_comparison["pct_descartadas"] = (
    filter_comparison["descartadas"] / filter_comparison["raw"].replace(0, np.nan)
)
display(filter_comparison.style.format({
    "raw": "{:.0f}", "con_template": "{:.0f}", "descartadas": "{:.0f}",
    "pct_descartadas": "{:.0%}",
}))
total_raw = filter_comparison["raw"].sum()
total_kept = filter_comparison["con_template"].sum()
print(f"\nTotal: {total_raw:.0f} señales raw -> {total_kept:.0f} con template ({total_kept/total_raw:.0%} retenidas)")

# --- Win Rate ---
pivot_winrate = comparison_df.pivot(index="ticker", columns="volume_ratio_threshold", values="winrate")
print("\n=== Win Rate por Ticker y Threshold ===")
display(pivot_winrate.style.format("{:.0%}").background_gradient(cmap="RdYlGn", vmin=0, vmax=1))

# --- Retorno Acumulado ---
pivot_cumret = comparison_df.pivot(index="ticker", columns="volume_ratio_threshold", values="cumulative_return")
print("\n=== Retorno Acumulado por Ticker y Threshold ===")

def color_returns(val):
    if isinstance(val, (int, float)) and not np.isnan(val):
        if val > 0:
            return "background-color: #27ae60; color: white"
        elif val < 0:
            return "background-color: #e74c3c; color: white"
    return ""

display(pivot_cumret.style.format("{:+.1%}").map(color_returns))

# --- Drawdown por Trade (peor) ---
pivot_worst_dd = comparison_df.pivot(index="ticker", columns="volume_ratio_threshold", values="worst_trade_drawdown")
print("\n=== Peor Drawdown por Trade por Ticker y Threshold ===")
display(pivot_worst_dd.style.format("{:+.1%}").background_gradient(cmap="RdYlGn", vmin=-0.3, vmax=0))

# --- Drawdown promedio por Trade ---
pivot_avg_dd = comparison_df.pivot(index="ticker", columns="volume_ratio_threshold", values="avg_trade_drawdown")
print("\n=== Drawdown Promedio por Trade por Ticker y Threshold ===")
display(pivot_avg_dd.style.format("{:+.1%}").background_gradient(cmap="RdYlGn", vmin=-0.2, vmax=0))

# --- Max Drawdown del Activo ---
pivot_asset_dd = comparison_df.pivot(index="ticker", columns="volume_ratio_threshold", values="asset_max_drawdown")
print("\n=== Max Drawdown del Activo (periodo completo) ===")
display(pivot_asset_dd.iloc[:, :1].style.format("{:+.1%}").background_gradient(cmap="RdYlGn", vmin=-0.9, vmax=-0.2))

# --- Patrones Detectados ---
pivot_patterns = comparison_df.pivot(index="ticker", columns="volume_ratio_threshold", values="n_patterns")
print("\n=== Patrones Detectados por Ticker y Threshold ===")
display(pivot_patterns.style.format("{:.0f}").background_gradient(cmap="Blues"))

## 7. Gráfico comparativo agregado

In [ ]:
matplotlib.use("Agg")

traded_df = comparison_df[comparison_df["n_trades"] > 0]

agg_by_threshold = traded_df.groupby("volume_ratio_threshold").agg({
    "n_patterns": "sum",
    "n_trades": "sum",
    "winrate": "mean",
    "cumulative_return": "mean",
    "worst_trade_drawdown": "min",
    "avg_trade_drawdown": "mean",
}).reset_index()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

plot_configs = [
    ("n_patterns", "Total Patrones", "{:.0f}"),
    ("n_trades", "Total Trades", "{:.0f}"),
    ("winrate", "Avg Win Rate", "{:.0%}"),
    ("cumulative_return", "Avg Retorno Acum.", "{:+.1%}"),
    ("worst_trade_drawdown", "Peor DD por Trade", "{:+.1%}"),
    ("avg_trade_drawdown", "Avg DD por Trade", "{:+.1%}"),
]

colors = ["#3498db", "#e67e22", "#27ae60"]

for ax, (col, title, fmt) in zip(axes, plot_configs):
    bars = ax.bar(
        agg_by_threshold["volume_ratio_threshold"].astype(str),
        agg_by_threshold[col],
        color=colors,
    )
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("volume_ratio_threshold")
    for bar, val in zip(bars, agg_by_threshold[col]):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            fmt.format(val),
            ha="center",
            va="bottom" if val >= 0 else "top",
            fontsize=10,
        )

plt.suptitle("VCP con Trend Template - Comparacion por Threshold", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
chart_path = project_root / "experiments" / "threshold_comparison_with_template.png"
fig.savefig(str(chart_path), dpi=110, bbox_inches="tight")
plt.close(fig)
print(f"Grafico guardado en: {chart_path}")

display(agg_by_threshold.style.format({
    "n_patterns": "{:.0f}",
    "n_trades": "{:.0f}",
    "winrate": "{:.0%}",
    "cumulative_return": "{:+.1%}",
    "worst_trade_drawdown": "{:+.1%}",
    "avg_trade_drawdown": "{:+.1%}",
}))

## 8. Resumen: impacto del Trend Template

In [ ]:
# Resumen del filtrado por Trend Template
total_by_threshold = comparison_df.groupby("volume_ratio_threshold").agg({
    "n_signals_raw": "sum",
    "n_signals": "sum",
    "n_filtered": "sum",
    "n_trades": "sum",
}).reset_index()

total_by_threshold["pct_retained"] = (
    total_by_threshold["n_signals"] / total_by_threshold["n_signals_raw"]
)

print("=== Impacto del Trend Template por Threshold ===\n")
display(total_by_threshold.rename(columns={
    "n_signals_raw": "sin_filtro",
    "n_signals": "con_template",
    "n_filtered": "descartadas",
    "n_trades": "trades_finales",
    "pct_retained": "pct_retenidas",
}).style.format({
    "sin_filtro": "{:.0f}",
    "con_template": "{:.0f}",
    "descartadas": "{:.0f}",
    "trades_finales": "{:.0f}",
    "pct_retenidas": "{:.1%}",
}))

# Resumen global
print("\n=== Resumen Global ===")
total_raw = int(comparison_df["n_signals_raw"].sum())
total_kept = int(comparison_df["n_signals"].sum())
total_discarded = int(comparison_df["n_filtered"].sum())
print(f"Senales VCP totales (sin filtro):     {total_raw}")
print(f"Senales retenidas (con template):     {total_kept} ({total_kept/total_raw:.1%})")
print(f"Senales descartadas por template:     {total_discarded} ({total_discarded/total_raw:.1%})")

## 9. Conclusiones

Para explorar los resultados en detalle:
```bash
mlflow ui --backend-store-uri mlruns/
```

### Diferencias vs experimento original (`run_vcp_experiment.ipynb`)

1. **Trend Template activo**: cada señal VCP se valida contra las 7 condiciones de Minervini.
   Señales en activos fuera de Etapa 2 se descartan antes del agrupamiento en patrones.

2. **Métricas de drawdown**:
   - `worst_trade_drawdown`: peor caída desde pico durante un trade (por ticker).
   - `avg_trade_drawdown`: drawdown promedio por trade (por ticker).
   - `asset_max_drawdown`: drawdown máximo del activo en todo el periodo (contexto de riesgo).

### Estructura de MLflow

Cada parent run contiene:
- **summaries/**: CSV con métricas por ticker (incluyendo drawdown) + CSV con todos los trades
- Métricas agregadas: winrate, cumulative return, drawdown, señales filtradas

Cada child run contiene:
- **pattern_plots/**: gráficos de cada patrón VCP detectado
- **trade_plots/**: gráficos de simulación de cada trade
- **tables/**: CSV con detalle de trades (incluyendo max_drawdown por trade)
- Métricas individuales incluyendo `n_signals_raw`, `n_filtered_by_template`